# **Bluestock_MF_Capstone_Project**
### **Phase 2:  Data Cleaning & SQL Database Design**

---
* **Prepared By:** Aalok Kumar Singh
* **Date:** June 3, 2026
* **Contact:** aalokkrsingh27@gmail.com
* **Role:** Data Analytics Intern
---

#### **STEP 1: Clean NAV History**

##### **NAV History Data Cleaning**

This notebook cleans and validates the mutual fund NAV history dataset.

Cleaning Steps:
- Parse dates
- Sort data
- Remove duplicates
- Handle missing NAV values
- Validate NAV values

------

##### **Load Dataset**

In [1]:
# Import libraries
import pandas as pd
import numpy as np 

In [2]:
# Load raw NAV history data
nav_df = pd.read_csv('../data/raw/02_nav_history.csv')

In [3]:
# Display first 5 rows of the dataset
nav_df.head() 

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


------

##### **Check Dataset Info**

In [4]:
# Check number of rows and columns
nav_df.shape 

(46000, 3)

In [5]:
# Check data types and non-null counts
nav_df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB


In [6]:
# Check for missing values
nav_df.isnull().sum() 

amfi_code    0
date         0
nav          0
dtype: int64

------

##### **Parse Dates**

In [7]:
# Convert 'date' column to datetime format 
nav_df['date'] = pd.to_datetime(nav_df['date']) 

In [8]:
# Check data type of 'date' column after conversion
nav_df['date'].info() 

<class 'pandas.Series'>
RangeIndex: 46000 entries, 0 to 45999
Series name: date
Non-Null Count  Dtype         
--------------  -----         
46000 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 359.5 KB


-------

##### **Sort Values**

In [9]:
# Sort dataset by 'amfi_code' and 'date' for time series analysis 
nav_df = nav_df.sort_values(by=['amfi_code', 'date'])

In [10]:
# Display cleaned dataset after processing  
nav_df 

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


-------

##### **Remove Duplicates**

In [11]:
# Store number of rows before removing duplicates
before_duplicates = nav_df.shape[0]   

In [12]:
# Remove duplicate rows based on all columns 
nav_df = nav_df.drop_duplicates()

In [13]:
# Store number of rows after removing duplicates
after_duplicates = nav_df.shape[0] 

In [14]:
# Print number of duplicates removed
print("Duplicates Removed: ", before_duplicates - after_duplicates) 

Duplicates Removed:  0


--------

##### **Validate NAV > 0**

In [15]:
# Identify rows with invalid NAV values (<= 0) is smaller or equals to zero
invalid_nav = nav_df[nav_df['nav'] <= 0]


In [16]:
print("Invalid NAV Rows: ")
print(invalid_nav.shape[0]) # Print number of rows with invalid NAV values

Invalid NAV Rows: 
0


In [17]:
# Remove rows with invalid NAV values (<= 0)
nav_df = nav_df[nav_df['nav'] > 0] 

------

##### **Handle Missing NAV Values**

In [18]:
# Fill remaining missing NAV values using forward fill and backward fill within each 'amfi_code' group
nav_df['nav'] = nav_df.groupby('amfi_code')['nav'].ffill().bfill() 

-------

##### **Final NULL Check**

In [19]:
# Check for any remaining missing values after filling
nav_df.isnull().sum() 

amfi_code    0
date         0
nav          0
dtype: int64

-------

##### **Save Clean Dataset**

In [20]:
nav_df.to_csv("../data/processed/clean_nav.csv", index =False)
print("Cleaned dataset saved sucessfully.")

Cleaned dataset saved sucessfully.


------

#### **STEP 2: Clean Transactions Dataset**

In [21]:
# Load raw investor transactions data
tx_df = pd.read_csv('../data/raw/08_investor_transactions.csv')

In [22]:
# Display first 5 rows of the dataset
tx_df.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


-------

##### **Dataset Info**

In [23]:
# Check number of rows and columns
tx_df.shape

(32778, 13)

In [24]:
# Checking data types and non-null counts
tx_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  str    
 1   transaction_date    32778 non-null  str    
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  str    
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  str    
 6   city                32778 non-null  str    
 7   city_tier           32778 non-null  str    
 8   age_group           32778 non-null  str    
 9   gender              32778 non-null  str    
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  str    
 12  kyc_status          32778 non-null  str    
dtypes: float64(1), int64(2), str(10)
memory usage: 3.3 MB


In [25]:
# Check for missing values
tx_df.isnull().sum()

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

------

##### **Fix Dates**

In [26]:
# Convert 'transaction_date' column to datetime format
tx_df['transaction_date'] = pd.to_datetime(tx_df['transaction_date'])


In [27]:
# Check data type of 'transaction_date' column after conversion
tx_df['transaction_date']

0       2024-01-01
1       2024-01-01
2       2024-01-01
3       2024-01-01
4       2024-01-01
           ...    
32773   2025-05-30
32774   2025-05-30
32775   2025-05-30
32776   2025-05-30
32777   2025-05-30
Name: transaction_date, Length: 32778, dtype: datetime64[us]

--------

##### **Standardize Transaction Types**

In [28]:
# Removing leading/trailing whitespace and standardizing case in 'transaction_type' column
tx_df['transaction_type'] = (tx_df['transaction_type'].str.strip().str.title())

-------

##### **Validate Amount > 0**

In [29]:
# Identify rows with invalid transaction amounts (<= 0)
invalid_amount = tx_df[tx_df['amount_inr'] <= 0]

In [30]:
# Print number of rows with invalid transaction amounts
print("Invalid Amount Rows: ")
print(invalid_amount.shape[0])

Invalid Amount Rows: 
0


In [31]:
# Remove rows with invalid transaction amounts (<= 0)
tx_df = tx_df[tx_df['amount_inr'] > 0]

------

##### **Validate KYC Status**

In [32]:
# Check distribution of KYC status in the dataset 
print(tx_df['kyc_status'].value_counts())

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


--------

##### **Save Clean Transactions**

In [33]:
# Save cleaned transactions dataset to processed folder
tx_df.to_csv("../data/processed/clean_transactions.csv", index =False)
print("Cleaned dataset saved sucessfully.")

Cleaned dataset saved sucessfully.


--------

#### **STEP 3: Clean Performance Dataset**

In [34]:
# Load raw scheme performance data
perf_df = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [35]:
# Display first 5 rows of the dataset
perf_df.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


--------

#####  **Check Data Types**

In [36]:
# Checking data types and non-null counts
perf_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     str    
 2   fund_house          40 non-null     str    
 3   category            40 non-null     str    
 4   plan                40 non-null     str    
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ratio_pct   4

In [37]:
# Checking the shape of the dataset
print(perf_df.shape)

(40, 19)


---------

#### **Convert Return Columns To Numeric**

In [38]:
# Converting return columns to numeric, coercing errors to NaN for invalid entries
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'alpha',
    'beta',
    'sharpe_ratio',
    'sortino_ratio'
]

for col in return_cols:
    perf_df[col] = pd.to_numeric(
        perf_df[col],
        errors='coerce'
    )

------

#### **Flag Negative Sharpe Ratios**

In [39]:
# Checking the number of funds with negative Sharpe ratio 
negative_sharpe = perf_df[
    perf_df['sharpe_ratio'] < 0
]

In [40]:
# Print number of funds with negative Sharpe ratio
print("Funds With Negative Sharpe Ratio:")
print(len(negative_sharpe))

Funds With Negative Sharpe Ratio:
0


--------

#### **Validate Expense Ratio**

In [41]:
# Identify rows with invalid expense ratios (less than 0.1% or greater than 2.5%)
invalid_expense = perf_df[
    (perf_df['expense_ratio_pct'] < 0.1) |
    (perf_df['expense_ratio_pct'] > 2.5)
]

In [42]:
# Print number of rows with invalid expense ratios
print("Invalid Expense Ratios:")
print(len(invalid_expense))

Invalid Expense Ratios:
0


------

#### **Save Clean Performance Dataset**

In [43]:
perf_df.to_csv(
    "../data/processed/clean_performance.csv",
    index=False
)

print("Performance dataset cleaned.")

Performance dataset cleaned.


-------


##### **STEP 4: Create SQLite Schema**
Location: 
*sql\schema.sql*

-------

Why Star Schema?

Because:

dimensions = descriptive data
facts = measurable metrics

------

#### **STEP 5: Create Database Loading Notebook Section**

In [44]:
import sqlite3
from sqlalchemy import create_engine

------

##### **Create SQLite Database**

In [45]:
engine = create_engine(
    'sqlite:///../data/db/bluestock_mf.db'
)

print("Database created successfully.")

Database created successfully.


-------

##### **Load Cleaned CSVs**

In [46]:
fund_df = pd.read_csv("../data/raw/01_fund_master.csv")

fund_df.to_sql(
    'dim_fund',
    engine,
    if_exists='replace',
    index=False
)

print("dim_fund loaded.")

dim_fund loaded.


--------

#### **Load NAV Data**

In [47]:
nav_df.to_sql(
    'fact_nav',
    engine,
    if_exists='replace',
    index=False
)

print("fact_nav loaded.")

fact_nav loaded.


-------

##### **Load Transactions**

In [48]:
tx_df.to_sql(
    'fact_transactions',
    engine,
    if_exists='replace',
    index=False
)

print("fact_transactions loaded.")

fact_transactions loaded.


-------

##### **Load Performance**

In [49]:
perf_df.to_sql(
    'fact_performance',
    engine,
    if_exists='replace',
    index=False
)

print("fact_performance loaded.")

fact_performance loaded.


--------

##### **Verify Tables**

In [50]:
conn = sqlite3.connect("../data/db/bluestock_mf.db")

query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql(query, conn)

tables

,name
0,dim_fund
1,fact_nav
2,fact_transactions
3,fact_performance


-------

#### **STEP 6: Write SQL Queries** 

------

#### **STEP 7: Run Queries** 

In [51]:
query = """
SELECT
    state,
    COUNT(*) AS total_transactions
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC;
"""

result = pd.read_sql(query, conn)

result.head()

,state,total_transactions
0,Punjab,2965
1,Madhya Pradesh,2931
2,Tamil Nadu,2806
3,Gujarat,2780
4,West Bengal,2748


In [55]:
# The 6 raw datasets we haven't officially "cleaned" yet
remaining_files = [
    '03_aum_by_fund_house.csv',
    '04_monthly_sip_inflows.csv',
    '05_category_inflows.csv',
    '06_industry_folio_count.csv',
    '09_portfolio_holdings.csv',
    '10_benchmark_indices.csv'
]

print("Starting batch cleaning...")

for file in remaining_files:
    # 1. Load the raw file
    df_temp = pd.read_csv(f'../data/raw/{file}')
    
    # 2. Basic Cleaning: Drop nulls and duplicates
    before_rows = df_temp.shape[0]
    df_temp = df_temp.dropna().drop_duplicates()
    after_rows = df_temp.shape[0]
    
    # 3. Create a clean filename (e.g., 'clean_aum_by_fund_house.csv')
    clean_name = f"clean_{file.split('_', 1)[1]}" 
    
    # 4. Save to processed folder
    df_temp.to_csv(f'../data/processed/{clean_name}', index=False)
    
    print(f"Saved: {clean_name} | Rows: {before_rows} -> {after_rows}")

    # 5. Save the 01_fund_master.csv file as well
    clean_fund_master = pd.read_csv('../data/raw/01_fund_master.csv').dropna().drop_duplicates()
    clean_fund_master.to_csv('../data/processed/clean_fund_master.csv', index=False)

Starting batch cleaning...
Saved: clean_aum_by_fund_house.csv | Rows: 90 -> 90
Saved: clean_monthly_sip_inflows.csv | Rows: 48 -> 36
Saved: clean_category_inflows.csv | Rows: 144 -> 144
Saved: clean_industry_folio_count.csv | Rows: 21 -> 21
Saved: clean_portfolio_holdings.csv | Rows: 322 -> 322
Saved: clean_benchmark_indices.csv | Rows: 8050 -> 8050
